When i ran the # CELL 14: Training Visualization & Checkpoint Utilities
================================================================================
  TRAINING VISUALIZATION
================================================================================


📊 Training Metrics Summary:
   Min Train Loss:  0.0513 (Epoch 25)
   Min Val Loss:    0.2278 (Epoch 25)
   Max Train Acc:   97.15% (Epoch 25)
   Max Val Acc:     92.39% (Epoch 22)
   Total GPU Mem:   5281 MB peak

✅ Good generalization: Train-Val gap = 4.8%

✅ Training curves saved to: /kaggle/working/MobilePlantViT/training_curves.png

============================================================
  CHECKPOINT VERIFICATION
============================================================

📁 Best Model Checkpoint:

🔍 Verifying checkpoint: /kaggle/working/MobilePlantViT/checkpoints/MobilePlantViT-Base_best.pth
------------------------------------------------------------
   ❌ Verification failed: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

📁 Last Model Checkpoint:
   File size: 14.87 MB
   ✅ Checkpoint exists

✅ Checkpoint info saved to: /kaggle/working/MobilePlantViT/checkpoint_info.json

✅ Checkpoint utilities defined:
   • load_checkpoint(path) - Load model from checkpoint
   • verify_checkpoint(path) - Verify checkpoint integrity
   • resume_training(path, model, optimizer) - Resume training

   There i found verification failed. Help me fix that

The issue is that PyTorch 2.6+ changed the default behavior of `torch.load()` to `weights_only=True` for security reasons, but your checkpoint contains numpy scalar objects in the training history that aren't allowed by default.

Here's the fix - update the `load_checkpoint` and `verify_checkpoint` functions in CELL 14 to explicitly set `weights_only=False`:



In [ ]:
// Update the load_checkpoint function in CELL 14

def load_checkpoint(checkpoint_path, device=None):
    """
    Load a comprehensive checkpoint and reconstruct the model.
    
    Args:
        checkpoint_path: Path to the checkpoint file
        device: Device to load model to (auto-detected if None)
    
    Returns:
        Dictionary containing:
        - 'model': Reconstructed and loaded model
        - 'optimizer': Optimizer with loaded state (if training_mode=True)
        - 'scheduler': Scheduler with loaded state (if exists)
        - 'epoch': Last completed epoch
        - 'best_val_acc': Best validation accuracy
        - 'history': Training history
        - 'config': Model configuration
        - 'metadata': Additional metadata
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Loading checkpoint from: {checkpoint_path}")
    
    # Load checkpoint with weights_only=False to allow numpy arrays in history
    # This is safe for checkpoints we created ourselves
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Extract metadata
    model_type = checkpoint.get('model_type', 'unknown')
    model_name = checkpoint.get('model_name', 'Unknown Model')
    num_classes = checkpoint.get('num_classes', 38)
    
    print(f"   Model: {model_name}")
    print(f"   Type: {model_type}")
    print(f"   Classes: {num_classes}")
    print(f"   Best Val Acc: {checkpoint.get('best_val_acc', 0)*100:.2f}%")
    print(f"   Epoch: {checkpoint.get('epoch', 0) + 1}")
    
    # Reconstruct model based on type
    if model_type == 'mobileplant_vit':
        model_config = checkpoint.get('model_config', None)
        
        if model_config is not None:
            # Reconstruct from saved config
            print(f"   Reconstructing from saved config...")
            config = MobilePlantViTConfig(**model_config)
            loaded_model = MobilePlantViT(config)
        else:
            # Fallback to base variant
            print(f"   ⚠️  No config found, using base variant")
            loaded_model = mobileplant_vit_base(num_classes=num_classes)
    
    elif model_type == 'mobilenet_v2':
        loaded_model = models.mobilenet_v2(weights=None)
        loaded_model.classifier[1] = nn.Linear(
            loaded_model.classifier[1].in_features, 
            num_classes
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # Load state dict
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    
    print(f"   ✅ Model loaded successfully!")
    
    # Prepare result
    result = {
        'model': loaded_model,
        'epoch': checkpoint.get('epoch', 0),
        'best_val_acc': checkpoint.get('best_val_acc', 0),
        'best_val_loss': checkpoint.get('best_val_loss', float('inf')),
        'history': checkpoint.get('history', {}),
        'config': checkpoint.get('model_config', {}),
        'class_names': checkpoint.get('class_names', []),
        'num_classes': num_classes,
        'training_config': checkpoint.get('training_config', {}),
        'timestamp': checkpoint.get('timestamp', 'unknown'),
        'is_best': checkpoint.get('is_best', False),
    }
    
    # Include optimizer state if available
    if 'optimizer_state_dict' in checkpoint:
        result['optimizer_state_dict'] = checkpoint['optimizer_state_dict']
    
    # Include scheduler state if available
    if 'scheduler_state_dict' in checkpoint:
        result['scheduler_state_dict'] = checkpoint['scheduler_state_dict']
    
    return result


def verify_checkpoint(checkpoint_path):
    """
    Verify checkpoint integrity and display contents.
    
    Args:
        checkpoint_path: Path to checkpoint file
    
    Returns:
        True if checkpoint is valid, False otherwise
    """
    print(f"\n🔍 Verifying checkpoint: {checkpoint_path}")
    print("-" * 60)
    
    try:
        # Load with weights_only=False since our checkpoints contain numpy arrays
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        
        # Required keys
        required_keys = ['model_state_dict', 'model_type', 'num_classes']
        optional_keys = [
            'model_name', 'model_config', 'optimizer_state_dict',
            'scheduler_state_dict', 'epoch', 'best_val_acc', 'best_val_loss',
            'history', 'class_names', 'training_config', 'timestamp', 'is_best'
        ]
        
        # Check required keys
        missing_required = [k for k in required_keys if k not in checkpoint]
        if missing_required:
            print(f"   ❌ Missing required keys: {missing_required}")
            return False
        
        # Display checkpoint contents
        print(f"   ✅ Required keys present")
        print(f"\n   📋 Checkpoint Contents:")
        
        for key in checkpoint.keys():
            value = checkpoint[key]
            if isinstance(value, dict):
                if key == 'model_state_dict':
                    print(f"      • {key}: {len(value)} parameters")
                elif key == 'history':
                    epochs = len(value.get('train_loss', []))
                    print(f"      • {key}: {epochs} epochs recorded")
                else:
                    print(f"      • {key}: dict with {len(value)} entries")
            elif isinstance(value, list):
                print(f"      • {key}: list with {len(value)} items")
            elif isinstance(value, (int, float, str, bool)):
                print(f"      • {key}: {value}")
            else:
                print(f"      • {key}: {type(value).__name__}")
        
        # Verify model can be loaded
        print(f"\n   🔄 Testing model reconstruction...")
        result = load_checkpoint(checkpoint_path)
        
        # Quick inference test
        test_input = torch.randn(1, 3, 224, 224).to(next(result['model'].parameters()).device)
        with torch.no_grad():
            output = result['model'](test_input)
        
        print(f"   ✅ Model inference test passed")
        print(f"      Output shape: {tuple(output.shape)}")
        
        return True
        
    except Exception as e:
        print(f"   ❌ Verification failed: {e}")
        return False


def resume_training(checkpoint_path, model, optimizer, scheduler=None, device=None):
    """
    Resume training from a checkpoint.
    
    Args:
        checkpoint_path: Path to checkpoint file
        model: Model instance (must match checkpoint architecture)
        optimizer: Optimizer instance
        scheduler: Optional scheduler instance
        device: Device to use
    
    Returns:
        Dictionary with loaded state and starting epoch
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Resuming training from: {checkpoint_path}")
    
    # Load with weights_only=False for our checkpoints
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if available
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    # Get training state
    start_epoch = checkpoint.get('epoch', 0) + 1
    best_val_acc = checkpoint.get('best_val_acc', 0)
    history = checkpoint.get('history', {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': [], 'epoch_time': [], 'gpu_memory_mb': []
    })
    
    print(f"   ✅ Resumed from epoch {start_epoch}")
    print(f"   Best Val Acc so far: {best_val_acc*100:.2f}%")
    
    return {
        'model': model,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'start_epoch': start_epoch,
        'best_val_acc': best_val_acc,
        'history': history,
    }



The key change is adding `weights_only=False` to all `torch.load()` calls:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)



This is safe because:
1. You created these checkpoints yourself during training
2. The numpy scalars in the history (like GPU memory values) are the reason for the error
3. You're loading from your own working directory, not untrusted sources

After making this change, re-run CELL 14 and the verification should pass.

now when i ran the # CELL 15: Test Set Evaluation
================================================================================
  TEST SET EVALUATION
================================================================================
📂 Loading best model checkpoint...
📂 Loading checkpoint from: /kaggle/working/MobilePlantViT/checkpoints/MobilePlantViT-Base_best.pth
---------------------------------------------------------------------------
UnpicklingError                           Traceback (most recent call last)
/tmp/ipykernel_47/2913362363.py in <cell line: 0>()
     14 
     15 # Use the checkpoint loading utility
---> 16 checkpoint_result = load_checkpoint(best_model_path, device=device)
     17 eval_model = checkpoint_result['model']
     18 eval_model.eval()

/tmp/ipykernel_47/1612941012.py in load_checkpoint(checkpoint_path, device)
    150 
    151     # Load checkpoint
--> 152     checkpoint = torch.load(checkpoint_path, map_location=device)
    153 
    154     # Extract metadata

/usr/local/lib/python3.11/dist-packages/torch/serialization.py in load(f, map_location, pickle_module, weights_only, mmap, **pickle_load_args)
   1468                         )
   1469                     except pickle.UnpicklingError as e:
-> 1470                         raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
   1471                 return _load(
   1472                     opened_zipfile,

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

The issue is that CELL 14 defines the `load_checkpoint` function, but you need to re-run it after making the fix. The error shows that the `load_checkpoint` function in memory still has the old code without `weights_only=False`.

Looking at your notebook, I can see that CELL 14 already has the updated `load_checkpoint` function with `weights_only=False`. However, it seems like the cell wasn't re-run after the update, or there's a version mismatch.

Let me check the exact line causing the issue. The error points to line 152 in the `load_checkpoint` function, which shows:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)



This is **missing** the `weights_only=False` parameter. Your CELL 14 code in the notebook shows the correct version, but you need to **re-run CELL 14** to update the function definition in memory.

**Quick Fix - Re-run CELL 14:**

Simply re-run CELL 14 (Training Visualization & Checkpoint Utilities) which contains the corrected `load_checkpoint` function with `weights_only=False`:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)



**Alternative - If you want to fix it directly in CELL 15:**

Add this code at the beginning of CELL 15 before calling `load_checkpoint`:



In [ ]:
# CELL 15: Test Set Evaluation - Quick fix for checkpoint loading

# Add this at the very beginning of CELL 15, before the existing code:

# Override load_checkpoint to fix weights_only issue
def load_checkpoint(checkpoint_path, device=None):
    """Load checkpoint with weights_only=False for compatibility."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Loading checkpoint from: {checkpoint_path}")
    
    # Load with weights_only=False (safe for our own checkpoints)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    model_type = checkpoint.get('model_type', 'unknown')
    model_name_loaded = checkpoint.get('model_name', 'Unknown Model')
    num_classes = checkpoint.get('num_classes', 38)
    
    print(f"   Model: {model_name_loaded}")
    print(f"   Type: {model_type}")
    print(f"   Classes: {num_classes}")
    print(f"   Best Val Acc: {checkpoint.get('best_val_acc', 0)*100:.2f}%")
    print(f"   Epoch: {checkpoint.get('epoch', 0) + 1}")
    
    if model_type == 'mobileplant_vit':
        model_config = checkpoint.get('model_config', None)
        if model_config is not None:
            print(f"   Reconstructing from saved config...")
            config = MobilePlantViTConfig(**model_config)
            loaded_model = MobilePlantViT(config)
        else:
            print(f"   ⚠️  No config found, using base variant")
            loaded_model = mobileplant_vit_base(num_classes=num_classes)
    elif model_type == 'mobilenet_v2':
        loaded_model = models.mobilenet_v2(weights=None)
        loaded_model.classifier[1] = nn.Linear(loaded_model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    print(f"   ✅ Model loaded successfully!")
    
    return {
        'model': loaded_model,
        'epoch': checkpoint.get('epoch', 0),
        'best_val_acc': checkpoint.get('best_val_acc', 0),
        'best_val_loss': checkpoint.get('best_val_loss', float('inf')),
        'history': checkpoint.get('history', {}),
        'config': checkpoint.get('model_config', {}),
        'class_names': checkpoint.get('class_names', []),
        'num_classes': num_classes,
    }

# Now the rest of CELL 15 continues as normal...



**Recommended Action:**
1. Go back to **CELL 14** 
2. **Re-run CELL 14** (this will update the `load_checkpoint` function in memory)
3. Then run **CELL 15** again

The `load_checkpoint` function in CELL 14 already has the correct fix - it just needs to be executed again to update the function definition in the Python runtime.

This is the complete code file i worked on. Analyse the complete code in each cell along with its outputs and tell me how it is performed. What is the exact flow of this.

# Complete Analysis of MobilePlantViT Training Notebook

This is a comprehensive **end-to-end deep learning training pipeline** for plant disease classification using a custom lightweight hybrid CNN-Transformer architecture called **MobilePlantViT**.

---

## 📋 High-Level Flow Overview



In [ ]:
┌─────────────────────────────────────────────────────────────────────────────┐
│                        TRAINING PIPELINE FLOW                                │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  [1] Setup & Imports ──► [2] Load Dataset ──► [3] Helper Functions          │
│           │                     │                      │                     │
│           ▼                     ▼                      ▼                     │
│  [4] Transforms ──────► [5] Create Datasets ──► [6] DataLoaders             │
│                                                        │                     │
│                                                        ▼                     │
│  [7] Model Config ──► [8] Visualize Samples ──► [9] Model Setup             │
│           │                                            │                     │
│           ▼                                            ▼                     │
│  [10] Pipeline Verify ──► [11] Summary ──► [12] Loss/Optimizer              │
│                                                        │                     │
│                                                        ▼                     │
│  [13] TRAINING LOOP ──► [14] Visualize Curves ──► [15] Test Eval            │
│           │                     │                      │                     │
│           ▼                     ▼                      ▼                     │
│  [16] Confusion Matrix ──► [17] Model Compare ──► [18] Mislabel Detection   │
│                                                        │                     │
│                                                        ▼                     │
│  [19] Model Export ──► [20] Validation Suite ──► [21] Final Summary         │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘



---

## 🔍 Detailed Cell-by-Cell Analysis

### **CELL 1: Setup & Imports**

**Purpose:** Initialize the training environment with all required libraries.

**Flow:**


In [ ]:
1. Import standard libraries (os, sys, json, datetime, time)
2. Import data science libraries (numpy, matplotlib, PIL)
3. Import deep learning libraries (torch, torchvision, tqdm)
4. Detect environment (Kaggle vs Local)
5. Add MobilePlantViT source code to Python path
6. Import custom MobilePlantViT components:
   - Model classes (MobilePlantViT, MobilePlantViTConfig)
   - Factory functions (mobileplant_vit_tiny/small/base/large)
   - Building blocks (GhostConv, CoordAtt, LinearDifferentialAttention, etc.)
7. Set random seeds for reproducibility (RANDOM_SEED = 42)
8. Display setup summary



**Key Output Variables:**
- `MOBILEPLANT_VIT_AVAILABLE`: Boolean flag
- `RANDOM_SEED`: 42

---

### **CELL 2: Load Preprocessed Dataset**

**Purpose:** Load the pre-split PlantVillage dataset from Kaggle.

**Flow:**


In [ ]:
1. Construct path to Kaggle dataset
2. Create working directory (/kaggle/working/MobilePlantViT)
3. Load dataset_metadata.json containing:
   - NUM_CLASSES: 38
   - CLASS_NAMES: List of 38 plant disease classes
   - IMAGENET_MEAN/STD: Normalization statistics
   - split_stats: Image counts per split
4. Set directory paths (train_dir, val_dir, test_dir)
5. Configure DataLoader settings (BATCH_SIZE=64, NUM_WORKERS=4)



**Dataset Structure:**


In [ ]:
plantvillage-color-preprocessed/
├── train/           # ~70% of images
├── val/             # ~15% of images
├── test/            # ~15% of images
└── dataset_metadata.json



---

### **CELL 3: Helper Functions**

**Purpose:** Define utility functions used throughout the pipeline.

**Functions Defined:**

| Function | Purpose |
|----------|---------|
| `print_section(title)` | Formatted section headers |
| `count_images_in_dir(directory)` | Count images per class |
| `calculate_class_weights(class_distribution)` | Inverse-frequency weights for imbalanced data |
| `create_weighted_sampler(dataset, class_distribution)` | Balanced batch sampling |
| `denormalize(tensor)` | Reverse ImageNet normalization for visualization |

---

### **CELL 4: Data Transformations**

**Purpose:** Define image preprocessing pipelines.

**Training Transform (9 steps):**


In [ ]:
transforms.Compose([
    Resize(256),                    # Resize shortest side
    RandomCrop(224),                # Random 224x224 crop
    RandomHorizontalFlip(0.5),      # 50% flip
    RandomRotation(30°),            # ±30° rotation
    RandomAffine(...),              # Translation, scale, shear
    ColorJitter(...),               # Brightness, contrast, saturation, hue
    RandomPerspective(...),         # Perspective distortion
    ToTensor(),                     # Convert to [0,1] tensor
    Normalize(ImageNet)             # ImageNet statistics
])



**Val/Test Transform (3 steps):**


In [ ]:
transforms.Compose([
    Resize((224, 224)),             # Fixed size
    ToTensor(),
    Normalize(ImageNet)
])



---

### **CELL 5: Create Datasets**

**Purpose:** Create PyTorch ImageFolder datasets.



In [ ]:
train_dataset = ImageFolder(train_dir, transform=transform_train)
val_dataset = ImageFolder(val_dir, transform=transform_val_test)
test_dataset = ImageFolder(test_dir, transform=transform_val_test)



**Verification:** Checks that class order matches metadata.

---

### **CELL 6: Create DataLoaders**

**Purpose:** Create batched data loaders with optional weighted sampling.

**Decision Logic:**


In [ ]:
if imbalance_ratio > 2.0 and balancing_strategy == 'weighted_sampling':
    → Create WeightedRandomSampler (oversample minority classes)
else:
    → Use standard shuffle



**DataLoader Configuration:**
- `batch_size=64`
- `num_workers=4`
- `pin_memory=True`
- `prefetch_factor=2`
- `persistent_workers=True`

---

### **CELL 7: Model Configuration**

**Purpose:** Central configuration hub for all training hyperparameters.

**Model Selection:**


In [ ]:
MODEL_TYPE = 'mobileplant_vit'  # or 'mobilenet_v2'
MODEL_VARIANT = 'base'          # tiny/small/base/large/custom



**Variant Specifications:**

| Variant | embed_dim | num_heads | Parameters | Use Case |
|---------|-----------|-----------|------------|----------|
| tiny | 128 | 4 | ~220K | Edge devices |
| small | 192 | 6 | ~490K | Mobile apps |
| **base** | 256 | 8 | ~867K | Default |
| large | 384 | 12 | ~1.9M | High accuracy |

**Training Configuration:**


In [ ]:
TRAINING_CONFIG = {
    'num_epochs': 25,
    'optimizer': 'adamw',
    'learning_rate': 3e-4,
    'weight_decay': 0.01,
    'scheduler': 'cosine_warmup',
    'warmup_epochs': 3,
    'gradient_clip_max_norm': 1.0,
    'use_mixed_precision': True,
    'early_stopping_patience': 7,
}



**Validation:** Checks `embed_dim % num_heads == 0`, learning rate bounds, etc.

---

### **CELL 8: Sample Visualization**

**Purpose:** Display augmented training samples for visual verification.



In [ ]:
images, labels = next(iter(train_loader))
# Display 2x4 grid of augmented images with class names



---

### **CELL 9: Model Setup**

**Purpose:** Instantiate the model and display comprehensive summary.

**MobilePlantViT Creation:**


In [ ]:
if MODEL_VARIANT == 'base':
    model = mobileplant_vit_base(num_classes=NUM_CLASSES)
elif MODEL_VARIANT == 'custom':
    config = MobilePlantViTConfig(**CUSTOM_CONFIG)
    model = MobilePlantViT(config)



**Model Summary Generated:**
- Total/trainable parameters
- Model size in MB
- Parameter breakdown by stage (CNN/Transition/Transformer/Classifier)
- Forward pass timing benchmark
- Parameter budget check (must be < 5M)

**Comparison Mode:** If enabled, also creates MobileNetV2 baseline.

---

### **CELL 10: Pipeline Verification**

**Purpose:** Comprehensive pre-training validation.

**Tests Performed:**

| Test | Description |
|------|-------------|
| Basic Forward Pass | Verify output shape `(B, NUM_CLASSES)` |
| `get_logits()` | Returns raw logits (for CrossEntropyLoss) |
| `forward_features()` | Returns feature vector before classifier |
| `get_intermediate_outputs()` | Outputs at each stage |
| Consistency Check | `forward() == softmax(get_logits())` |
| Dimensional Flow | Track tensor shapes through network |
| Speed Benchmark | 50 runs for statistical accuracy |
| Gradient Flow | Verify gradients reach all parameters |
| Memory Usage | Report GPU memory consumption |

**Dimensional Flow Example:**


In [ ]:
Input:              (B, 3, 224, 224)
After GhostConv:    (B, 64, 112, 112)
After Fused-IR:     (B, 128, 56, 56)
After CoordAtt:     (B, 128, 56, 56)
After PatchEmbed:   (B, 196, 256)
After Transformer:  (B, 196, 256)
After GAP:          (B, 256)
Output:             (B, 38)



---

### **CELL 11: Pipeline Summary**

**Purpose:** Display complete configuration before training.

Summarizes:
- Dataset configuration
- Data pipeline settings
- Transformation details
- Model architecture

---

### **CELL 12: Loss Function & Optimizer Setup**

**Purpose:** Configure all training components.

**Critical Note:**


In [ ]:
# MobilePlantViT output methods:
model.forward(x)      # Returns PROBABILITIES (after softmax)
model.get_logits(x)   # Returns RAW LOGITS (before softmax)

# CrossEntropyLoss expects raw logits, so training uses:
logits = model.get_logits(images)
loss = criterion(logits, labels)



**Components Created:**

| Component | Configuration |
|-----------|---------------|
| Loss | `CrossEntropyLoss` (weighted if imbalanced) |
| Optimizer | `AdamW` (lr=3e-4, weight_decay=0.01) |
| Scheduler | Cosine warmup (3 epoch warmup → cosine decay) |
| AMP Scaler | `torch.amp.GradScaler('cuda')` |
| Early Stopping | Patience=7, monitor val_acc |

**Learning Rate Schedule Visualization:**


In [ ]:
LR
 ^
 |    /‾‾‾‾‾‾‾‾‾‾‾\
 |   /             \
 |  /               \
 | /                 \__min_lr
 +-----------------------> Epochs
   [warmup]  [cosine decay]



---

### **CELL 13: Training Loop** ⭐ (Core Training)

**Purpose:** Main training loop with all modern best practices.

**Per-Epoch Flow:**



In [ ]:
# TRAINING PHASE
model.train()
for batch in train_loader:
    images, labels = batch.to(device)
    optimizer.zero_grad(set_to_none=True)
    
    with torch.amp.autocast('cuda'):
        logits = model.get_logits(images)  # NOT forward()!
        loss = criterion(logits, labels)
    
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()  # Step-based scheduler

# VALIDATION PHASE
model.eval()
with torch.no_grad():
    for batch in val_loader:
        logits = model.get_logits(images)
        loss = criterion(logits, labels)
        # Accumulate metrics

# END OF EPOCH
save_checkpoint(last_model_path)
if val_acc > best_val_acc:
    save_checkpoint(best_model_path)
if early_stopper(val_acc):
    break



**Checkpoint Contents:**


In [ ]:
{
    'model_state_dict': ...,
    'model_config': ...,
    'optimizer_state_dict': ...,
    'scheduler_state_dict': ...,
    'epoch': ...,
    'best_val_acc': ...,
    'history': ...,
    'num_classes': 38,
    'class_names': [...],
    'training_config': {...},
    'timestamp': ...,
}



---

### **CELL 14: Training Visualization & Checkpoint Utilities**

**Purpose:** Visualize training progress and provide checkpoint utilities.

**Visualizations:**
1. Loss curves (train vs val)
2. Accuracy curves (train vs val)
3. Learning rate schedule
4. Epoch duration bar chart

**Utility Functions:**


In [ ]:
load_checkpoint(path)          # Reconstruct model from checkpoint
verify_checkpoint(path)        # Validate checkpoint integrity
resume_training(path, model)   # Resume from checkpoint



---

### **CELL 15: Test Set Evaluation**

**Purpose:** Evaluate best model on held-out test set.

**Metrics Computed:**
- Test loss and accuracy
- Top-3 and Top-5 accuracy
- Per-class accuracy
- Inference speed (throughput, latency)

**Output:**


In [ ]:
TEST_RESULTS = {
    'test_accuracy': 0.95,
    'top3_accuracy': 0.99,
    'top5_accuracy': 0.995,
    'throughput_imgs_per_sec': 450,
    'per_class_accuracy': {...},
}



---

### **CELL 16: Confusion Matrix & Classification Report**

**Purpose:** Detailed classification metrics.

**Outputs:**
- Full confusion matrix heatmap
- Normalized confusion matrix (recall per class)
- Top 15 confusion pairs (most common misclassifications)
- Per-class precision/recall/F1
- Macro and weighted averages

---

### **CELL 17: Model Comparison**

**Purpose:** Compare MobilePlantViT vs MobileNetV2 baseline.

**Comparison Table:**
| Metric | MobilePlantViT | MobileNetV2 | Winner |
|--------|----------------|-------------|--------|
| Parameters | ~867K | ~3.5M | ✅ MPV |
| Test Accuracy | 95% | 96% | MNV2 |
| Inference Time | 2ms | 8ms | ✅ MPV |
| Efficiency Score | High | Low | ✅ MPV |

**Visualization:** 4-panel comparison chart

---

### **CELL 18: Mislabeled Image Detection**

**Purpose:** Find potentially mislabeled training images.

**Algorithm:**


In [ ]:
for each image in train_set (without augmentation):
    if prediction != label AND loss is high:
        → Flag as potentially mislabeled



**Output:**
- Top 100 suspicious samples
- Visualization of top 10 candidates
- `mislabeled_candidates.json`

---

### **CELL 19: Model Export for Deployment**

**Purpose:** Export trained model in multiple formats.

**Export Formats:**

| Format | File | Use Case |
|--------|------|----------|
| PyTorch Full | `_full_checkpoint.pth` | Resume training |
| PyTorch State | `_state_dict.pth` | Lightweight loading |
| TorchScript | `_traced.pt` | C++/Mobile deployment |
| ONNX | `.onnx` | Cross-platform (TensorRT, CoreML) |

**Deployment Metadata:**


In [ ]:
{
    "preprocessing": {
        "input_size": [224, 224],
        "normalization": {"mean": [...], "std": [...]}
    },
    "class_mapping": {"0": "Apple___healthy", ...}
}



**Inference Examples Generated:**
- `inference_example_pytorch.py`
- `inference_example_onnx.py`

---

### **CELL 20: End-to-End Validation Suite**

**Purpose:** Automated verification of entire pipeline.

**Validation Categories:**

| Category | Checks |
|----------|--------|
| Setup | MobilePlantViT imports, device, random seed |
| Data | Dataset loading, DataLoaders, class count, batch shape |
| Model | Instantiation, parameters < 5M, device, forward pass, get_logits() |
| Training | Completion, improvement, no NaN loss, checkpoint saved |
| Evaluation | Test accuracy, quality, Top-K hierarchy |
| Export | Directory, TorchScript, ONNX |
| Files | All expected output files present |

**Output:** `validation_report.json`

---

### **CELL 21: Final Summary**

**Purpose:** Comprehensive summary and next steps.

**Displays:**
- Model information
- Training results
- Test results
- Validation status
- Output files listing
- Recommendations for deployment
- Quick reference code snippets

---

## 🏗️ MobilePlantViT Architecture Flow



In [ ]:
Input Image (3, 224, 224)
         │
         ▼
┌─────────────────────────────────────┐
│     CNN STAGE (Local Features)      │
├─────────────────────────────────────┤
│  GhostConv (3→64, stride=2)         │  → (64, 112, 112)
│  Fused-IR Block (64→128, stride=2)  │  → (128, 56, 56)
│  Coordinate Attention               │  → (128, 56, 56)
└─────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────┐
│    TRANSITION (CNN → Transformer)   │
├─────────────────────────────────────┤
│  Patch Embedding (128→256)          │  → (196, 256)  [14×14 patches]
│  Positional Encoding                │  → (196, 256)
└─────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────┐
│  TRANSFORMER STAGE (Global Context) │
├─────────────────────────────────────┤
│  × 3 Transformer Blocks:            │
│    ├─ Linear Differential Attention │
│    ├─ Residual LayerNorm Block      │
│    └─ Bottleneck FFN                │
└─────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────┐
│       CLASSIFIER STAGE              │
├─────────────────────────────────────┤
│  Global Average Pooling             │  → (256,)
│  Classifier Head (256→38)           │  → (38,)
│  Softmax (in forward())             │  → probabilities
└─────────────────────────────────────┘



---

## 🔑 Key Design Decisions

### 1. **Dual Output Methods**


In [ ]:
model.forward(x)      # Returns probabilities (for inference)
model.get_logits(x)   # Returns logits (for training with CrossEntropyLoss)



### 2. **Mixed Precision Training**


In [ ]:
with torch.amp.autocast('cuda'):
    logits = model.get_logits(images)
    loss = criterion(logits, labels)
scaler.scale(loss).backward()



### 3. **Gradient Clipping for Transformers**


In [ ]:
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)



### 4. **Cosine Warmup Schedule**


In [ ]:
# Linear warmup for first 3 epochs
# Then cosine decay to min_lr



### 5. **Weighted Sampling for Imbalanced Data**


In [ ]:
if imbalance_ratio > 2.0:
    sampler = WeightedRandomSampler(weights, num_samples, replacement=True)



---

## 📊 Expected Outputs

After running the complete notebook:



In [ ]:
/kaggle/working/MobilePlantViT/
├── checkpoints/
│   ├── MobilePlantViT-Base_best.pth
│   └── MobilePlantViT-Base_last.pth
├── exports/
│   ├── mobileplant_vit_full_checkpoint.pth
│   ├── mobileplant_vit_state_dict.pth
│   ├── mobileplant_vit_traced.pt
│   ├── mobileplant_vit.onnx
│   ├── deployment_metadata.json
│   ├── inference_example_pytorch.py
│   └── inference_example_onnx.py
├── training_curves.png
├── confusion_matrix.png
├── confusion_matrix_normalized.png
├── model_comparison.png
├── mislabeled_visualization.png
├── test_results.json
├── classification_report.json
├── model_comparison.json
├── validation_report.json
├── mislabeled_candidates.json
├── checkpoint_info.json
└── final_summary.json



---

## 🎯 Summary

This notebook implements a **production-ready training pipeline** for plant disease classification with:

1. **Custom Architecture**: MobilePlantViT combining CNN efficiency with Transformer global understanding
2. **Modern Training Practices**: AMP, gradient clipping, cosine warmup, early stopping
3. **Comprehensive Evaluation**: Top-K accuracy, confusion matrix, per-class metrics
4. **Deployment Ready**: Multiple export formats with inference examples
5. **Quality Assurance**: Automated validation suite and mislabeled detection

Is my current code following the exact architecture flow right?

# Architecture Analysis: MobilePlantViT Training Notebook

After carefully analyzing your complete notebook against the architecture diagram, I can confirm:

## ✅ **Yes, your code correctly follows the architecture flow.**

The training notebook properly implements and utilizes the MobilePlantViT architecture as specified. Here's the verification:

---

## 🔍 Architecture Alignment Check

### 1. **CNN Stage (Local Features)** ✅

Your notebook imports and uses the correct blocks:


In [ ]:
from src.blocks import (
    GhostConv,                    # 3→64, stride=2 → (64, 112, 112)
    FusedInvertedResidualBlock,   # 64→128, stride=2 → (128, 56, 56)
    CoordAtt,                     # Attention → (128, 56, 56)
    ...
)



**Cell 10 (Pipeline Verification)** confirms this dimensional flow:


In [ ]:
stage_mapping = {
    'after_ghost_conv': 'After GhostConv',
    'after_fused_ir': 'After Fused-IR',
    'after_coord_att': 'After CoordAtt',
    ...
}



---

### 2. **Transition Stage (CNN → Transformer)** ✅

Correctly imports:


In [ ]:
PatchEmbedding,      # 128→256 → (196, 256)
PositionalEncoding,  # Adds position info → (196, 256)



**Cell 10** tracks this:


In [ ]:
'after_patch_embed': 'After PatchEmbed',
'after_pos_encoding': 'After PosEncode',



The expected 196 patches = 14×14 (from 56×56 feature map with implied 4×4 patch size):


In [ ]:
expected_patches = 196  # (224/16)^2 for typical patch size



---

### 3. **Transformer Stage (Global Context)** ✅

Correctly imports the custom attention mechanism:


In [ ]:
LinearDifferentialAttention,  # Novel attention (not standard MHSA)
ResidualLayerNormBlock,       # Pre-norm residual
BottleneckFFN,                # Efficient FFN



Configuration in **Cell 7** matches:


In [ ]:
VARIANT_SPECS = {
    'base': {
        'embed_dim': 256,
        'num_heads': 8,
        # num_transformer_blocks: 3 (default)
    },
}



---

### 4. **Classifier Stage** ✅

Correctly imports:


In [ ]:
GlobalAveragePooling,  # (B, 196, 256) → (B, 256)
ClassifierHead,        # (B, 256) → (B, 38)



---

## 🎯 Critical Implementation Detail: Dual Output Methods

Your notebook **correctly handles** the architecture's dual output design:

| Method | Output | Use Case |
|--------|--------|----------|
| `model.forward(x)` | Probabilities (after softmax) | Inference |
| `model.get_logits(x)` | Raw logits | Training with `CrossEntropyLoss` |

**Cell 12** documents this clearly:


In [ ]:
# MobilePlantViT output methods:
# model.forward(x)      # Returns PROBABILITIES (after softmax)
# model.get_logits(x)   # Returns RAW LOGITS (before softmax)



**Cell 13 (Training Loop)** correctly uses `get_logits()`:


In [ ]:
if MODEL_TYPE == 'mobileplant_vit':
    logits = model.get_logits(images)  # NOT forward()!
else:
    logits = model(images)
loss = criterion(logits, labels)



---

## 📊 Dimensional Flow Verification (Cell 10)

Your verification code tracks the exact flow:



In [ ]:
Stage                     Shape
─────────────────────────────────────────────────
Input                     (B, 3, 224, 224)
After GhostConv           (B, 64, 112, 112)    ✅
After Fused-IR            (B, 128, 56, 56)     ✅
After CoordAtt            (B, 128, 56, 56)     ✅
After PatchEmbed          (B, 196, 256)        ✅
After PosEncode           (B, 196, 256)        ✅
After Transformer         (B, 196, 256)        ✅
After GAP                 (B, 256)             ✅
Logits                    (B, 38)              ✅
Output (probs)            (B, 38)              ✅



---

## ✅ Summary

| Aspect | Status |
|--------|--------|
| CNN Stage blocks | ✅ Correct |
| Transition (Patch + Position) | ✅ Correct |
| Transformer (LDA + FFN) | ✅ Correct |
| Classifier (GAP + Head) | ✅ Correct |
| `forward()` returns probabilities | ✅ Correct |
| `get_logits()` returns raw logits | ✅ Correct |
| Training uses `get_logits()` | ✅ Correct |
| Dimensional flow | ✅ Verified |
| Variant configurations | ✅ Match architecture |

**Your notebook is correctly implementing and following the MobilePlantViT architecture flow.** The training loop, evaluation, and export all properly account for the dual-output design pattern.

In this file i ran on the base version. Now i want to change it and test with other versions. What all changes i need to do in this code?

# Changing MobilePlantViT Variants

To test different variants (tiny, small, large) instead of base, you only need to change **ONE line** in **Cell 7**.

---

## 🔧 Quick Change (Cell 7)

Find this line in Cell 7:



In [ ]:
// In Cell 7: Model Configuration

# For MobilePlantViT: select variant
# Options: 'tiny', 'small', 'base', 'large', 'custom'
MODEL_VARIANT = 'base'  # <-- CHANGE THIS LINE



Change it to one of these:

| Variant | Code | Parameters | Use Case |
|---------|------|------------|----------|
| **Tiny** | `MODEL_VARIANT = 'tiny'` | ~220K | Edge devices, IoT |
| **Small** | `MODEL_VARIANT = 'small'` | ~490K | Mobile apps |
| **Base** | `MODEL_VARIANT = 'base'` | ~867K | Default (current) |
| **Large** | `MODEL_VARIANT = 'large'` | ~1.9M | Maximum accuracy |

---

## 📋 Example: Testing All Variants

### For Tiny:


In [ ]:
// Cell 7 - Change this line
MODEL_VARIANT = 'tiny'



### For Small:


In [ ]:
// Cell 7 - Change this line
MODEL_VARIANT = 'small'



### For Large:


In [ ]:
// Cell 7 - Change this line
MODEL_VARIANT = 'large'



---

## ⚙️ Optional: Adjust Training Config for Different Variants

For **tiny/small** variants (smaller models), you might want to:



In [ ]:
// Cell 7 - Optional adjustments for smaller variants

TRAINING_CONFIG = {
    'num_epochs': 30,              # More epochs for smaller models
    'learning_rate': 5e-4,         # Slightly higher LR for smaller models
    # ... rest stays the same
}



For **large** variant (larger model), you might want to:



In [ ]:
// Cell 7 - Optional adjustments for large variant

TRAINING_CONFIG = {
    'num_epochs': 20,              # Fewer epochs (converges faster)
    'learning_rate': 2e-4,         # Slightly lower LR for stability
    'batch_size': 32,              # Reduce if GPU memory is limited
    # ... rest stays the same
}



---

## 🚀 After Changing: Run Order

1. **Re-run Cell 7** (Model Configuration) - to update the variant
2. **Re-run Cell 9** (Model Setup) - to create the new model
3. **Re-run Cells 10-21** sequentially - to train and evaluate

Or simply: **Restart kernel and run all cells** after making the change.

---

## 📊 Variant Comparison Summary

| Variant | embed_dim | num_heads | Params | Speed | Accuracy | Best For |
|---------|-----------|-----------|--------|-------|----------|----------|
| `tiny` | 128 | 4 | ~220K | ⚡⚡⚡ Fastest | Lower | Real-time, IoT |
| `small` | 192 | 6 | ~490K | ⚡⚡ Fast | Good | Mobile apps |
| `base` | 256 | 8 | ~867K | ⚡ Balanced | Better | Default choice |
| `large` | 384 | 12 | ~1.9M | Slower | Best | High accuracy |

---

## 💡 Pro Tip: Compare Results

After training each variant, the results are saved to different checkpoint files based on the model name. You can compare them by looking at:

- `test_results.json` - Test accuracy
- `model_comparison.json` - Full comparison metrics
- `final_summary.json` - Complete summary

IN my current code i want a confusion matrix similar to this (design & structure).
Give me proper updated code for the cell # CELL 16: Confusion Matrix & Classification Report
Keep other reports as it is. Only modify the confusion matrix

Looking at the reference confusion matrix image, I can see it has a cleaner design with:
- White/light background
- Class names on both axes (rotated on x-axis)
- Numbers displayed in cells
- Blue color intensity based on values
- Clear title with "Confusion Matrix - Test Set"

Here's the updated code for Cell 16:



In [ ]:
# ============================================================================
# CELL 16: Confusion Matrix & Classification Report
# ============================================================================
"""
Generate confusion matrix and detailed classification report for test results.
"""

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print_section("CONFUSION MATRIX & CLASSIFICATION REPORT")

# ============================================================================
# CONFUSION MATRIX (Styled like reference image)
# ============================================================================
cm = confusion_matrix(all_labels, all_preds)

# Create figure with appropriate size for 38 classes
fig_size = 20

fig, ax = plt.subplots(figsize=(fig_size, fig_size))

# Use a clean white/blue colormap similar to reference
cmap = sns.color_palette("Blues", as_cmap=True)

# Create heatmap with annotations
# For 38 classes, we can show numbers but with smaller font
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap=cmap,
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    square=True,
    ax=ax,
    cbar=True,
    cbar_kws={'shrink': 0.6, 'aspect': 30},
    annot_kws={'size': 5},  # Small font for numbers
    linewidths=0.5,
    linecolor='white'
)

# Set title
ax.set_title(f'Confusion Matrix - Test Set', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Predicted Class', fontsize=14, labelpad=10)
ax.set_ylabel('True Class', fontsize=14, labelpad=10)

# Rotate x-axis labels for readability (like reference image)
plt.xticks(rotation=90, ha='center', fontsize=6)
plt.yticks(rotation=0, fontsize=6)

# Adjust layout
plt.tight_layout()
plt.savefig(os.path.join(working_base, 'confusion_matrix.png'), dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

print(f"\n✅ Confusion matrix saved to: {os.path.join(working_base, 'confusion_matrix.png')}")

# ============================================================================
# NORMALIZED CONFUSION MATRIX (easier to interpret)
# ============================================================================
print(f"\n📊 Generating normalized confusion matrix...")

# Normalize by row (true labels) to show recall per class
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)  # Handle division by zero

fig2, ax2 = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(
    cm_normalized,
    annot=False,  # Too crowded with percentages for 38 classes
    fmt='.2f',
    cmap=cmap,
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    square=True,
    ax=ax2,
    vmin=0,
    vmax=1,
    cbar=True,
    cbar_kws={'shrink': 0.6, 'aspect': 30, 'label': 'Recall'},
    linewidths=0.5,
    linecolor='white'
)

ax2.set_title(f'Normalized Confusion Matrix (Row = Recall)\n{model_name} | Test Acc: {test_acc*100:.2f}%',
              fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Predicted Class', fontsize=14, labelpad=10)
ax2.set_ylabel('True Class', fontsize=14, labelpad=10)

plt.xticks(rotation=90, ha='center', fontsize=6)
plt.yticks(rotation=0, fontsize=6)

plt.tight_layout()
plt.savefig(os.path.join(working_base, 'confusion_matrix_normalized.png'), dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print(f"✅ Normalized confusion matrix saved to: {os.path.join(working_base, 'confusion_matrix_normalized.png')}")

# ============================================================================
# TOP CONFUSION PAIRS (Most common misclassifications)
# ============================================================================
print(f"\n🔍 Top 15 Confusion Pairs (Most Common Misclassifications):")
print("-" * 80)

# Find off-diagonal elements (misclassifications)
confusion_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'true_class': CLASS_NAMES[i],
                'pred_class': CLASS_NAMES[j],
                'count': int(cm[i, j]),  # Convert numpy int64 to Python int
                'true_idx': int(i),       # Convert to Python int
                'pred_idx': int(j)        # Convert to Python int
            })

# Sort by count descending
confusion_pairs.sort(key=lambda x: x['count'], reverse=True)

print(f"{'#':<4} {'True Class':<30} {'Predicted As':<30} {'Count':<8}")
print("-" * 80)
for idx, pair in enumerate(confusion_pairs[:15], 1):
    print(f"{idx:<4} {pair['true_class'][:28]:<30} {pair['pred_class'][:28]:<30} {pair['count']:<8}")

# ============================================================================
# CLASSIFICATION REPORT
# ============================================================================
print(f"\n📊 Classification Report:")
print("-" * 80)

report = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4,
    output_dict=True,
    zero_division=0  # Suppress warnings, set to 0 for undefined metrics
)

# Print summary metrics
print(f"\n   Macro Avg Precision:  {report['macro avg']['precision']:.4f}")
print(f"   Macro Avg Recall:     {report['macro avg']['recall']:.4f}")
print(f"   Macro Avg F1-Score:   {report['macro avg']['f1-score']:.4f}")
print(f"   Weighted Avg F1:      {report['weighted avg']['f1-score']:.4f}")

# Show per-class performance summary
print(f"\n📈 Per-Class Performance Summary:")
print("-" * 80)

# Create per-class dataframe for analysis
class_metrics = []
for class_name in CLASS_NAMES:
    if class_name in report:
        class_metrics.append({
            'class': class_name,
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1': report[class_name]['f1-score'],
            'support': report[class_name]['support']
        })

# Sort by F1 score
class_metrics.sort(key=lambda x: x['f1'])

# Show worst 5 classes
print(f"\n🔴 Worst Performing Classes (by F1-Score):")
print(f"{'Class':<40} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
print("-" * 86)
for m in class_metrics[:5]:
    print(f"{m['class'][:38]:<40} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1']:<12.4f} {m['support']:<10}")

# Show best 5 classes
print(f"\n🟢 Best Performing Classes (by F1-Score):")
print(f"{'Class':<40} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
print("-" * 86)
for m in class_metrics[-5:][::-1]:
    print(f"{m['class'][:38]:<40} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1']:<12.4f} {m['support']:<10}")

# Save classification report
report_path = os.path.join(working_base, 'classification_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=4)

# Save confusion pairs
confusion_pairs_path = os.path.join(working_base, 'top_confusion_pairs.json')
with open(confusion_pairs_path, 'w') as f:
    json.dump(confusion_pairs[:50], f, indent=4)

print(f"\n✅ Classification report saved to: {report_path}")
print(f"✅ Top confusion pairs saved to: {confusion_pairs_path}")

# Store for comparison
CLASSIFICATION_REPORT = report



## Key Changes Made:

1. **Cleaner design** - Removed shortened class name labels, using full `CLASS_NAMES` directly
2. **White background** - Added `facecolor='white'` to savefig
3. **Better annotations** - Shows actual numbers in cells with `annot=True, fmt='d'`
4. **Smaller annotation font** - `annot_kws={'size': 5}` for readability with 38 classes
5. **White grid lines** - `linewidths=0.5, linecolor='white'` for cell separation
6. **Higher DPI** - Changed to `dpi=300` for sharper output
7. **Proper axis labels** - Full class names on both axes, rotated 90° on x-axis
8. **Cleaner colorbar** - Adjusted shrink and aspect ratio
9. **Simplified title** - "Confusion Matrix - Test Set" like the reference

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print_section("CONFUSION MATRIX & CLASSIFICATION REPORT")

Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

Plot
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
xticklabels=train_dataset.classes,
yticklabels=train_dataset.classes)
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Classification report
print("\n📊 Detailed Classification Report:")
print(classification_report(all_labels, all_preds,
target_names=train_dataset.classes,
digits=4))

This was the code for that perfect confusion matrix output. So modify it accordingly.

Now I understand! You want to keep the simpler, cleaner code that produced the perfect confusion matrix. Here's the updated Cell 16 that maintains that exact confusion matrix style while keeping the additional reports:



In [ ]:
# ============================================================================
# CELL 16: Confusion Matrix & Classification Report
# ============================================================================
"""
Generate confusion matrix and detailed classification report for test results.
"""

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print_section("CONFUSION MATRIX & CLASSIFICATION REPORT")

# ============================================================================
# CONFUSION MATRIX
# ============================================================================
cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix (exact style from reference)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(working_base, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Confusion matrix saved to: {os.path.join(working_base, 'confusion_matrix.png')}")

# ============================================================================
# CLASSIFICATION REPORT
# ============================================================================
print("\n📊 Detailed Classification Report:")
print(classification_report(all_labels, all_preds, 
                           target_names=train_dataset.classes,
                           digits=4))

# ============================================================================
# ADDITIONAL ANALYSIS (Optional)
# ============================================================================

# Top Confusion Pairs
print(f"\n🔍 Top 15 Confusion Pairs (Most Common Misclassifications):")
print("-" * 80)

confusion_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'true_class': CLASS_NAMES[i],
                'pred_class': CLASS_NAMES[j],
                'count': int(cm[i, j]),
                'true_idx': int(i),
                'pred_idx': int(j)
            })

confusion_pairs.sort(key=lambda x: x['count'], reverse=True)

print(f"{'#':<4} {'True Class':<35} {'Predicted As':<35} {'Count':<8}")
print("-" * 80)
for idx, pair in enumerate(confusion_pairs[:15], 1):
    print(f"{idx:<4} {pair['true_class'][:33]:<35} {pair['pred_class'][:33]:<35} {pair['count']:<8}")

# Save classification report as JSON
report = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4,
    output_dict=True,
    zero_division=0
)

report_path = os.path.join(working_base, 'classification_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=4)

confusion_pairs_path = os.path.join(working_base, 'top_confusion_pairs.json')
with open(confusion_pairs_path, 'w') as f:
    json.dump(confusion_pairs[:50], f, indent=4)

print(f"\n✅ Classification report saved to: {report_path}")
print(f"✅ Top confusion pairs saved to: {confusion_pairs_path}")

# Store for comparison
CLASSIFICATION_REPORT = report



## Summary of Changes:

| Aspect | Your Original (Kept) | Added |
|--------|---------------------|-------|
| Figure size | `(20, 16)` ✅ | - |
| Heatmap style | `annot=True, fmt='d', cmap='Blues'` ✅ | - |
| X-axis rotation | `rotation=45, ha='right'` ✅ | - |
| Title | `'Confusion Matrix - Test Set'` ✅ | - |
| Labels | `train_dataset.classes` ✅ | - |
| Save to file | - | ✅ Added |
| Top confusion pairs | - | ✅ Added |
| JSON exports | - | ✅ Added |

The confusion matrix will now look **exactly** like your reference image! 🎯

Task 16 from the Current_task.md is completed. I have done it completely. Now we need to start doing the task 17: Create Inference Pipeline.
Now guide me with complete step by step instructions to complete that task from beginning to end.